# Orbit Wars — H9 4p model (python-only, gate-first)

Modelo **4p** baseado em ameaça forward per-enemy (`bots/pgs/threat.py`, H9 / DB 169/234/236).
**Python-only**: nada de Rust/maturin/GPU — o bot PGS+threat e `orbit_lite` são puro Python e
a avaliação roda no env OFICIAL `kaggle_environments`.

Disciplina (DB 234/235/236): probe → v1 FALHOU (aniquilação total) → v2 PASSOU o gate de 500 steps
(morte_4p 0.75→0.65 vs holdwave). Este notebook **re-gateia no env oficial** e só empacota a
submissão **se** morte_4p(H9) < morte_4p(holdwave). 2p fica congelado por construção
(`threat_value_4p` só dispara em 4p; em 2p o bot é o holdwave embarcado).

In [ ]:
# ======================
# CONFIG
# ======================
RUN_NAME = "ow_h9_4p_pyonly_v01"

# Gate (env oficial). 500 steps e obrigatorio; suba seeds se sobrar tempo.
GATE_SEEDS = 16
GATE_STEPS = 500

# Config do bot. 2p = holdwave embarcado (congelado); 4p ganha threat-value + reinforce.
SHIPPED_CONFIG = 'scripts="hold", wave_min_ships=60.0, wave_start_step=150'
H9_CONFIG      = SHIPPED_CONFIG + ", threat_value_4p=True"

# So empacota a submissao se o gate passar.
PACKAGE_IF_PASS = True
INSTALL_REQUIREMENTS = True   # pip (CPU). SEM build Rust.

In [ ]:
# ======================
# SETUP (python-only: sem Rust/maturin/GPU)
# ======================
import os, sys, subprocess, zipfile, shutil, json
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists() and Path("/kaggle/working").exists()
WORKING = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
OUT = WORKING / f"{RUN_NAME}_outputs"; OUT.mkdir(parents=True, exist_ok=True)

def find_project_root():
    # 1) ja descompactado em /kaggle/input
    for base in ([Path("/kaggle/input")] if IS_KAGGLE else [Path.cwd()]):
        for p in base.rglob("scripts/h9_4p_gate_kenv.py"):
            root = p.parents[1]
            if (root / "bots" / "pgs" / "threat.py").exists():
                return root
    # 2) zip do projeto em /kaggle/input
    if IS_KAGGLE:
        for z in Path("/kaggle/input").rglob("*.zip"):
            try:
                with zipfile.ZipFile(z) as zf:
                    if any(n.endswith("bots/pgs/threat.py") for n in zf.namelist()):
                        dest = WORKING / "project"; dest.mkdir(exist_ok=True)
                        zf.extractall(dest)
                        for p in dest.rglob("scripts/h9_4p_gate_kenv.py"):
                            return p.parents[1]
            except Exception as e:
                print("skip zip", z, e)
    raise FileNotFoundError("projeto nao encontrado (precisa do orbit-wars-lab com bots/pgs/threat.py)")

ROOT = find_project_root()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

if INSTALL_REQUIREMENTS and IS_KAGGLE:
    # instala deps python; NAO roda maturin/cargo (orbit_lite + bot sao puro python)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)

# o bot PGS e submission-safe (sem import de Rust). sanity:
import bots.pgs.agent, bots.pgs.threat
from kaggle_environments import make
print("imports OK (bots.pgs + kaggle_environments). Rust NAO e necessario.")

In [ ]:
# ======================
# GATE 4p no env OFICIAL (fonte unica: scripts/h9_4p_gate_kenv.py)
# ======================
from scripts.h9_4p_gate_kenv import run_config, SHIPPED, H9

seeds = list(range(3000, 3000 + GATE_SEEDS))
print(f"Gate: {GATE_SEEDS} seeds x {GATE_STEPS} steps, seat0 vs Producer x3 (env oficial)\n")

base = run_config("holdwave", SHIPPED, seeds, GATE_STEPS)
print(f"  holdwave : death={base['death_rate']:.3f} planets={base['mean_final_planets']:.2f}")
h9 = run_config("h9_threat", H9, seeds, GATE_STEPS)
print(f"  h9_threat: death={h9['death_rate']:.3f} planets={h9['mean_final_planets']:.2f}")

GATE_PASS = h9["death_rate"] < base["death_rate"]
verdict = {"pass": bool(GATE_PASS), "holdwave": base, "h9_threat": h9,
           "delta_death": h9["death_rate"] - base["death_rate"], "seeds": GATE_SEEDS, "steps": GATE_STEPS}
(OUT / "gate_verdict.json").write_text(json.dumps(verdict, indent=2))
print(f"\n=== H9 4p GATE: {'PASS' if GATE_PASS else 'FAIL'} "
      f"(holdwave {base['death_rate']:.3f} -> h9 {h9['death_rate']:.3f}) ===")

In [ ]:
# ======================
# EMPACOTAR SUBMISSAO (so se o gate passar)
# ======================
if GATE_PASS and PACKAGE_IF_PASS:
    tar = OUT / "submission_pgs_h9.tar.gz"
    r = subprocess.run([sys.executable, "scripts/package_pgs_submission.py",
                        "--pgs-config", H9_CONFIG, "--out", str(tar)],
                       cwd=ROOT, capture_output=True, text=True)
    print(r.stdout[-2000:]); print(r.stderr[-1000:])
    if tar.exists():
        # extrai o main.py/submission.py para a aba Output
        import tarfile
        with tarfile.open(tar) as tf:
            for n in tf.getnames():
                if n.endswith("main.py") or n.endswith("submission.py"):
                    tf.extract(n, OUT); print("extraido:", OUT / n)
        print(f"\nARTEFATO PRONTO: {tar}")
        print("Para submeter: baixe da aba Output e use a competicao, ou (local):")
        print(f"  kaggle competitions submit -c orbit-wars -f {tar.name} -m 'H9 4p threat (gate pass)'")
    else:
        print("ERRO: tarball nao gerado — ver stderr acima.")
else:
    print("Gate FALHOU (ou PACKAGE_IF_PASS=False): NAO empacotando submissao.")
    print("Disciplina: nao gasta slot de LB com bot que nao bate o holdwave em 4p.")

## Notas
- **Sem submissão silenciosa**: só empacota se o gate 4p oficial passar.
- **2p congelado**: `threat_value_4p` e o auto-enable de scripts só disparam em 4p; em 2p o bot é
  idêntico ao holdwave embarcado (`scripts="hold"`).
- Para retomar/escalar: suba `GATE_SEEDS`; se H9 não escalar, o próximo lever é H11 (atenção par-a-par, DB 170).